# [HTTP GET requests with the Python standard library](https://alexwlchan.net/2026/python-http-with-the-stdlib/)


In [1]:
import urllib.parse
import urllib.request

In [2]:
url = "https://booking.com"
params = {"name": "pentagon", "sides": "5"}
headers = {"User-Agent": "Shape-Sorter/1.0"}

u = urllib.parse.urlsplit(url)
query = urllib.parse.urlencode(params)
url = urllib.parse.urlunsplit(
    (u.scheme, u.netloc, u.path, query, u.fragment)
)

req = urllib.request.Request(url, headers=headers)

resp = urllib.request.urlopen(req)
print(resp.read())

b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n    <meta charset="utf-8">\r\n    <meta name="viewport" content="width=device-width, initial-scale=1">\r\n    <title></title>\r\n    <style>\r\n        body {\r\n            font-family: "Arial";\r\n        }\r\n    </style>\r\n    <script type="text/javascript" nonce="DoXoHRKvZzZNziE">\r\n        function reportChallengeError(log, challengeError) {\r\n    return;\r\n}\r\n\r\n        \r\n    </script>\r\n    <script type="text/javascript" nonce="DoXoHRKvZzZNziE">\r\n        window.awsWafCookieDomainList = [\'booking.com\'];\r\n    </script>\r\n    <script type="text/javascript" src="https://www.booking.com/__challenge_h78IRKX3kpQxScCExxShBNwRUlb/d8c14d4960ca/a18a4859af9c/challenge.js" nonce="DoXoHRKvZzZNziE"></script>\r\n    <script type="text/javascript" nonce="DoXoHRKvZzZNziE">\r\n        const maxLocationLength = 2047 - "&chal_t=1760623098515&force_referer=".length;\r\n        function searchStringWithNewParam(search, key, value) {\

In [3]:
import urllib.parse
import urllib.request


QueryParams = dict[str, str] | list[tuple[str, str]]
Headers = dict[str, str]


def build_request(
    url: str,
    *,
    params: QueryParams | None = None,
    headers: Headers | None = None
) -> urllib.request.Request:
    """
    Build a urllib Request, appending query parameters and attaching headers.
    """
    if params is not None:
        params_list = list(params.items()) if isinstance(params, dict) else params

        u = urllib.parse.urlsplit(url)
        query = urllib.parse.parse_qsl(u.query) + params_list
        new_query = urllib.parse.urlencode(query)
        url = urllib.parse.urlunsplit(
            (u.scheme, u.netloc, u.path, new_query, u.fragment)
        )

    req = urllib.request.Request(url, headers=headers or {})

    return req

In [4]:
import certifi
import ssl


def fetch_url(
    url: str,
    *,
    params: QueryParams | None = None,
    headers: Headers | None = None
) -> bytes:
    """
    Fetch the contents of a URL and return the body of the response.
    """
    req = build_request(url, params=params, headers=headers)
    
    ssl_context = ssl.create_default_context(cafile=certifi.where())

    with urllib.request.urlopen(req, context=ssl_context) as resp:
        data: bytes = resp.read()

    return data

## Downloading images with format-based file extensions

The mapping contains the four image formats I encounter in practice; it’s easy for me to add more if I try to download a newer format someday.

Then I wrote a function that takes an image URL and an “out prefix” (an initial guess at the path), downloads the image and choose a new file extension, and returns the final path:

In [5]:
def choose_filename_extension(content_type: str | None) -> str:
    """
    Choose a filename extension for an image downloaded with the given
    Content-Type header.
    """
    if content_type is None:
        raise ValueError(
            "no Content-Type header, cannot determine image format"
        )

    content_type_mapping = {
        "image/jpeg": "jpg",
        "image/png": "png",
        "image/gif": "gif",
        "image/webp": "webp",
    }

    try:
        return content_type_mapping[content_type]
    except KeyError:
        raise ValueError(f"unrecognised Content-Type header: {content_type}")


In [6]:
from pathlib import Path


def download_image(
    url: str,
    out_prefix: Path,
    *,
    params: QueryParams | None = None,
    headers: Headers | None = None,
) -> Path:
    """
    Download an image from the given URL to the target path, and return
    the path of the downloaded file.

    Add the appropriate file extension, based on the image's Content-Type.

    Throws a FileExistsError if you try to overwrite an existing file.
    """
    req = build_request(url, params=params, headers=headers)

    ssl_context = ssl.create_default_context(cafile=certifi.where())

    with urllib.request.urlopen(req, context=ssl_context) as resp:
        image_data: bytes = resp.read()

    image_format = choose_filename_extension(content_type=resp.headers["content-type"])

    out_path = out_prefix.with_suffix("." + image_format)
    out_path.parent.mkdir(exist_ok=True, parents=True)
    with open(out_path, "xb") as out_file:
        out_file.write(image_data)

    return out_path


In [10]:
# =========================
# BOOKING.COM PRICE SCRAPER
# JUPYTER-NOTEBOOK VERSION
# =========================

# INSTALL FIRST:
# !pip install playwright pandas nest_asyncio
# !playwright install

import asyncio
import nest_asyncio
from datetime import date, timedelta
from typing import Generator

import pandas as pd
from playwright.async_api import async_playwright

# Fix Jupyter async loop issue
nest_asyncio.apply()


# ==========================================
# CONFIG
# ==========================================

HOTEL_URL = (
    "https://www.booking.com/hotel/ch/"
    "swiss-diamond-olivella.html"
)

DAYS_AHEAD = 90
MIN_STAY = 1
MAX_STAY = 2

HEADLESS = False  # Set True for silent mode

STATIC_PARAMS = {
    "group_adults": "2",
    "group_children": "0",
    "no_rooms": "1",
    "req_adults": "2",
    "req_children": "0",
    "room1": "A,A",
}


# ==========================================
# DATE GENERATOR
# ==========================================

def generate_date_pairs(
    days_ahead: int,
    min_stay: int,
    max_stay: int,
) -> Generator[tuple[date, date], None, None]:

    today = date.today()

    for offset in range(days_ahead):

        checkin = today + timedelta(days=offset)

        for nights in range(min_stay, max_stay + 1):

            checkout = checkin + timedelta(days=nights)

            yield checkin, checkout


# ==========================================
# URL BUILDER
# ==========================================

def build_url(
    checkin: date,
    checkout: date,
) -> str:

    params = STATIC_PARAMS.copy()

    params["checkin"] = checkin.isoformat()
    params["checkout"] = checkout.isoformat()

    query_string = "&".join(
        f"{k}={v}" for k, v in params.items()
    )

    return f"{HOTEL_URL}?{query_string}"


# ==========================================
# PRICE EXTRACTION
# ==========================================

async def extract_price(page) -> str | None:

    selectors = [
        '[data-testid="price-and-discounted-price"]',
        '.prco-valign-middle-helper',
        '.bui-price-display__value',
    ]

    for selector in selectors:

        try:
            element = await page.query_selector(selector)

            if element:
                text = await element.inner_text()

                cleaned = (
                    text
                    .replace("\n", " ")
                    .replace("\xa0", " ")
                    .strip()
                )

                if cleaned:
                    return cleaned

        except Exception:
            pass

    return None


# ==========================================
# SCRAPER
# ==========================================

async def scrape_booking_prices() -> pd.DataFrame:

    rows: list[dict] = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=HEADLESS,
        )

        context = await browser.new_context(
            locale="en-US",
            user_agent=(
                "Mozilla/5.0 "
                "(Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/125.0 Safari/537.36"
            ),
        )

        page = await context.new_page()

        for checkin, checkout in generate_date_pairs(
            DAYS_AHEAD,
            MIN_STAY,
            MAX_STAY,
        ):

            url = build_url(checkin, checkout)

            print(
                f"Scraping: "
                f"{checkin} -> {checkout}"
            )

            try:

                await page.goto(
                    url,
                    wait_until="networkidle",
                    timeout=60000,
                )

                await page.wait_for_timeout(3000)

                price = await extract_price(page)

                rows.append({
                    "checkin": checkin.isoformat(),
                    "checkout": checkout.isoformat(),
                    "nights": (
                        checkout - checkin
                    ).days,
                    "price": price,
                    "url": url,
                })

                print(
                    f"  SUCCESS: {price}"
                )

            except Exception as exc:

                print(
                    f"  FAILED: {exc}"
                )

                rows.append({
                    "checkin": checkin.isoformat(),
                    "checkout": checkout.isoformat(),
                    "nights": (
                        checkout - checkin
                    ).days,
                    "price": None,
                    "url": url,
                })

            # polite throttling
            await page.wait_for_timeout(2000)

        await browser.close()

    df = pd.DataFrame(rows)

    return df


# ==========================================
# RUN
# ==========================================

df = await scrape_booking_prices()

print("\nDONE")
print(df.head())


# ==========================================
# SAVE
# ==========================================

df.to_csv(
    "booking_prices.csv",
    index=False,
)

print(
    "\nSaved to booking_prices.csv"
)

Scraping: 2026-05-26 -> 2026-05-27
  SUCCESS: CHF 340
Scraping: 2026-05-26 -> 2026-05-28
  SUCCESS: CHF 781
Scraping: 2026-05-27 -> 2026-05-28
  SUCCESS: CHF 298
Scraping: 2026-05-27 -> 2026-05-29
  SUCCESS: None
Scraping: 2026-05-28 -> 2026-05-29
  SUCCESS: None
Scraping: 2026-05-28 -> 2026-05-30
  SUCCESS: None
Scraping: 2026-05-29 -> 2026-05-30
  SUCCESS: None
Scraping: 2026-05-29 -> 2026-05-31
  SUCCESS: None
Scraping: 2026-05-30 -> 2026-05-31
  SUCCESS: CHF 475
Scraping: 2026-05-30 -> 2026-06-01
  SUCCESS: CHF 809
Scraping: 2026-05-31 -> 2026-06-01
  SUCCESS: CHF 383
Scraping: 2026-05-31 -> 2026-06-02
  SUCCESS: CHF 924
Scraping: 2026-06-01 -> 2026-06-02
  SUCCESS: CHF 571
Scraping: 2026-06-01 -> 2026-06-03
  SUCCESS: CHF 1,106
Scraping: 2026-06-02 -> 2026-06-03
  SUCCESS: CHF 571
Scraping: 2026-06-02 -> 2026-06-04
  SUCCESS: CHF 1,106
Scraping: 2026-06-03 -> 2026-06-04
  SUCCESS: CHF 571
Scraping: 2026-06-03 -> 2026-06-05
  SUCCESS: CHF 1,106
Scraping: 2026-06-04 -> 2026-06-05
  

In [12]:
# ==============================
# BOOKING PRICE WATCHER
# CHECK EVERY 2 HOURS
# JUPYTER VERSION
# ==============================

# INSTALL FIRST:
# !pip install playwright pandas nest_asyncio
# !playwright install

import asyncio
import nest_asyncio
from datetime import datetime

import pandas as pd
from playwright.async_api import async_playwright

# Jupyter async fix
nest_asyncio.apply()


# ======================================
# CONFIG
# ======================================

URL = (
    "https://www.booking.com/hotel/ch/"
    "swiss-diamond-olivella.html"
    "?group_adults=2"
    "&group_children=0"
    "&no_rooms=1"
    "&req_adults=2"
    "&req_children=0"
    "&room1=A,A"
    "&checkin=2026-05-27"
    "&checkout=2026-05-28"
)

CHECK_INTERVAL_SECONDS = 60 * 60 * 2  # 2 hours

CSV_FILE = "booking_price_history.csv"

HEADLESS = True


# ======================================
# EXTRACT PRICE
# ======================================

async def extract_price(page) -> str | None:

    selectors = [
        '[data-testid="price-and-discounted-price"]',
        '.prco-valign-middle-helper',
        '.bui-price-display__value',
    ]

    for selector in selectors:

        try:
            element = await page.query_selector(selector)

            if element:

                text = await element.inner_text()

                cleaned = (
                    text
                    .replace("\n", " ")
                    .replace("\xa0", " ")
                    .strip()
                )

                if cleaned:
                    return cleaned

        except Exception:
            pass

    return None


# ======================================
# SCRAPE SINGLE PRICE
# ======================================

async def scrape_price(page) -> str | None:

    await page.goto(
        URL,
        wait_until="networkidle",
        timeout=60000,
    )

    await page.wait_for_timeout(3000)

    price = await extract_price(page)

    return price


# ======================================
# SAVE RESULT
# ======================================

def save_result(price: str | None):

    now = datetime.now().isoformat()

    row = pd.DataFrame([
        {
            "timestamp": now,
            "price": price,
        }
    ])

    try:

        existing = pd.read_csv(CSV_FILE)

        updated = pd.concat(
            [existing, row],
            ignore_index=True,
        )

    except FileNotFoundError:

        updated = row

    updated.to_csv(
        CSV_FILE,
        index=False,
    )


# ======================================
# WATCHER LOOP
# ======================================

async def monitor_price():

    last_price = None

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=HEADLESS,
        )

        context = await browser.new_context(
            locale="en-US",
            user_agent=(
                "Mozilla/5.0 "
                "(Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/125.0 Safari/537.36"
            ),
        )

        page = await context.new_page()

        while True:

            try:

                print(
                    "\nChecking:",
                    datetime.now()
                )

                price = await scrape_price(page)

                print("Current price:", price)

                save_result(price)

                # detect change
                if (
                    last_price is not None
                    and price != last_price
                ):

                    print(
                        "\nPRICE CHANGED!"
                    )

                    print(
                        f"{last_price} -> {price}"
                    )

                last_price = price

            except Exception as exc:

                print("ERROR:", exc)

            print(
                f"\nSleeping for "
                f"{CHECK_INTERVAL_SECONDS / 3600} hours..."
            )

            await asyncio.sleep(
                CHECK_INTERVAL_SECONDS
            )


# ======================================
# START
# ======================================

await monitor_price()


Checking: 2026-05-26 13:18:59.652158
Current price: CHF 298

Sleeping for 2.0 hours...


CancelledError: 